# Car Price Prediction

Predicting the selling price of a used car from its specs.
Dataset: https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho


## 1. Load the data

Downloaded automatically with kagglehub.


In [ ]:
!pip install kagglehub -q

import kagglehub, os
path = kagglehub.dataset_download('nehalbirla/vehicle-dataset-from-cardekho')
print(os.listdir(path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(os.path.join(path, 'car data.csv'))
df.head()


## 2. Quick look at the data


In [ ]:
print(df.shape)
df.info()
print(df.isnull().sum())
df.describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Selling_Price'], kde=True, ax=axes[0]).set_title('selling price')
sns.scatterplot(x='Kms_Driven', y='Selling_Price', data=df, ax=axes[1]).set_title('km driven vs price')
plt.show()

plt.figure(figsize=(12,6))
sns.countplot(x='Fuel_Type', data=df)
plt.show()


## 3. Feature engineering and encoding

The name column has the brand as the first word, extracting that.
Car age usually matters more than the year itself, so I made an age column.
Encoding the categorical columns with LabelEncoder.


In [ ]:
df['brand'] = df['Car_Name'].str.split().str[0]
df['age'] = 2026 - df['Year']
df = df.drop(columns=['Car_Name', 'Year'])

from sklearn.preprocessing import LabelEncoder
for col in ['Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'brand']:
    df[col + '_enc'] = LabelEncoder().fit_transform(df[col].astype(str))

df = df.drop(columns=['Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'brand'])
df.head()


## 4. Training

Split the data, scale it, then try 3 models and compare them.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df.drop(columns=['Selling_Price'])
y = df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=42)
}

scores = {}
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted[name] = model
    scores[name] = {
        'R2': round(r2_score(y_test, pred), 4),
        'MAE': round(mean_absolute_error(y_test, pred), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 2)
    }

pd.DataFrame(scores).T


## 5. Best model and saving

Picking the model with the highest R2, plotting actual vs predicted, then saving with joblib.


In [ ]:
best_name = pd.DataFrame(scores).T['R2'].idxmax()
best_model = fitted[best_name]
print('best model:', best_name)

pred = best_model.predict(X_test)
plt.figure(figsize=(7,7))
sns.scatterplot(x=y_test, y=pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('actual price')
plt.ylabel('predicted price')
plt.show()


In [ ]:
os.makedirs('model_artifacts', exist_ok=True)
joblib.dump(best_model, 'model_artifacts/car_model.pkl')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')
joblib.dump(X.columns.tolist(), 'model_artifacts/features.pkl')
os.listdir('model_artifacts')
